# Relative RMSE — All Datasets and Models

Single notebook that computes the per-step **relative RMSE** for the first few autoregressive
rollout steps across every (dataset × model) pair under `logs/official/runs/`.

The "classical solver" (analytical / physics-informed reference) differs per dataset:

- `ac` (Allen-Cahn 1D)  → `ac` model
- `kdv` (KdV 1D)        → `kdv` model
- `ac_2d` (Allen-Cahn 2D) → `ac_2d` model
- `ckdv`, `fene_v2`     → no classical solver run

For each dataset we evaluate `N_EVAL` test trajectories and report `mean ± std` of the
relative RMSE at steps 1–5 (configurable below).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.prediction.helpers import (  # noqa: E402
    compute_rel_rmse_rollout,
    discover_models,
    load_test_data_nd,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT  : {ROOT}")
print(f"Device: {device}")

In [ ]:
# Per-dataset configuration.
# Keys:
#   data_glob       — HDF5 glob (relative to ROOT)
#   n_spatial_dims  — 1 for 1D fields, 2 for 2D
#   classical_key   — name of the classical/analytical solver run, or None
#   n_eval          — number of test trajectories to roll out
DATASETS = {
    "ac": {
        "data_glob": "data/allen_cahn/*.hdf5",
        "n_spatial_dims": 1,
        "classical_key": "ac",
        "n_eval": 10,
    },
    "kdv": {
        "data_glob": "data/kdv/*.hdf5",
        "n_spatial_dims": 1,
        "classical_key": "kdv",
        "n_eval": 10,
    },
    "ckdv": {
        "data_glob": "data/fput/*.hdf5",
        "n_spatial_dims": 1,
        "classical_key": None,
        "n_eval": 10,
    },
    "fene_v2": {
        "data_glob": "data/fene/*.hdf5",
        "n_spatial_dims": 1,
        "classical_key": None,
        "n_eval": 10,
    },
    "ac_2d": {
        "data_glob": "data/allen_cahn_2d/*1000_seed0.hdf5",
        "n_spatial_dims": 2,
        "classical_key": "ac_2d",
        "n_eval": 10,
    },
}

# Display labels — the classical solver entries vary per dataset, so each is keyed by run name.
DISPLAY_NAMES = {
    "fno": "FNO",
    "onsagernet1d_lowrank": "OnsagerNet",
    "res_onsagernet": "Res-OnsagerNet",
    "res_onsagernet_2d": "Res-OnsagerNet",
    "s_onsagernet": "S-OnsagerNet",
    "ac": "Allen-Cahn (Classical solver)",
    "kdv": "KdV (Classical solver)",
    "ac_2d": "Allen-Cahn 2D (Classical solver)",
    "ac_realV": "S-OnsagerNet w/ real $V$",
    "kdv_realV": "S-OnsagerNet w/ real $V$",
    "ac_2d_realV": "S-OnsagerNet w/ real $V$",
}

RUNS_BASE = ROOT / "logs/official/runs"
REPORT_STEPS = [1, 2, 3, 4, 5]
MAX_STEPS = max(REPORT_STEPS)
print(f"Reporting relative RMSE at steps {REPORT_STEPS} (MAX_STEPS={MAX_STEPS})")

In [ ]:
# Run rollouts for every (dataset, model) pair. Stored as nested dict
# results[dataset][model_name] = rel_rmse array of shape (N_eval, MAX_STEPS+1).
results: dict = {}

for ds_key, cfg in DATASETS.items():
    print(f"\n========== {ds_key} ==========")
    test_data, t_coord, x_coord, y_coord = load_test_data_nd(
        str(ROOT / cfg["data_glob"]),
        n_spatial_dims=cfg["n_spatial_dims"],
    )
    print(f"Test set shape: {tuple(test_data.shape)}")

    N_test = test_data.shape[0]
    n_eval = min(cfg["n_eval"], N_test)
    sample_idxs = np.linspace(0, N_test - 1, n_eval, dtype=int)
    eval_data = test_data[sample_idxs]
    print(f"Evaluating {n_eval} trajectories at indices {sample_idxs.tolist()}")

    models = discover_models(RUNS_BASE / ds_key, root=ROOT, device=device)
    ds_results: dict = {}
    for exp_name, m_info in models.items():
        print(f"  rollout '{exp_name}' ...")
        ds_results[exp_name] = compute_rel_rmse_rollout(m_info["model"], eval_data, max_steps=MAX_STEPS, device=device)
    results[ds_key] = ds_results

print("\nAll rollouts complete.")

In [ ]:
# Build a single tidy DataFrame: rows = (dataset, model), columns = step k.
rows = []
for ds_key, ds_res in results.items():
    classical_key = DATASETS[ds_key]["classical_key"]
    # preferred display order: classical → fno → onsagernet1d_lowrank → res_* → s_onsagernet → others
    order = [classical_key, "fno", "onsagernet1d_lowrank", "res_onsagernet", "res_onsagernet_2d", "s_onsagernet"]
    seen = set()
    ordered_keys = []
    for k in order:
        if k is not None and k in ds_res and k not in seen:
            ordered_keys.append(k)
            seen.add(k)
    for k in ds_res:
        if k not in seen:
            ordered_keys.append(k)
            seen.add(k)

    for exp_name in ordered_keys:
        arr = ds_res[exp_name]  # (N_eval, MAX_STEPS+1)
        row = {
            "dataset": ds_key,
            "model": exp_name,
            "display": DISPLAY_NAMES.get(exp_name, exp_name),
        }
        for s in REPORT_STEPS:
            row[f"step_{s}_mean"] = arr[:, s].mean()
            row[f"step_{s}_std"] = arr[:, s].std()
        rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# Pretty per-dataset tables (mean ± std).
for ds_key in DATASETS:
    sub = df[df["dataset"] == ds_key]
    if sub.empty:
        continue
    label_w = max(sub["display"].str.len().max() + 2, 22)
    col_w = 22
    print(f"\n=== {ds_key} ===")
    header = f"{'Model':<{label_w}}" + "".join(f"{'Step ' + str(s):>{col_w}}" for s in REPORT_STEPS)
    print(header)
    print("-" * len(header))
    for _, row in sub.iterrows():
        line = f"{row['display']:<{label_w}}"
        for s in REPORT_STEPS:
            line += f"{row[f'step_{s}_mean']:.4f} ± {row[f'step_{s}_std']:.4f}".rjust(col_w)
        print(line)

In [ ]:
# Save to CSV for reuse / paper tables.
out_dir = ROOT / "figs/prediction_table"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "rel_rmse_first5_all_datasets.csv"
df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv.relative_to(ROOT)}")